# The 2026 NeuroGolf Championship

## Score: 104.36

In [1]:
from pathlib import Path
import json
import tempfile
import zipfile
from typing import Dict, List, Tuple

import numpy as np
import onnx
import onnxruntime as ort

In [2]:
DATA_DIR = Path("data")

OUTPUT_DIR = Path("baseline_submission")
ZIP_NAME = Path("submission.zip")
LIMIT = 0
WRITE_ALL_TASKS = True

BATCH = 1
CHANNELS = 10
HEIGHT = 30
WIDTH = 30
DTYPE = onnx.TensorProto.FLOAT
GRID_SHAPE = [BATCH, CHANNELS, HEIGHT, WIDTH]
IR_VERSION = 10
OPSET_IMPORTS = [onnx.helper.make_opsetid("", 10)]

In [3]:
def one_hot_grid(grid: List[List[int]]) -> np.ndarray:
    tensor = np.zeros((BATCH, CHANNELS, HEIGHT, WIDTH), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            tensor[0, color, r, c] = 1.0
    return tensor


def make_conv_model(weights: np.ndarray, kernel_size: int) -> onnx.ModelProto:
    x = onnx.helper.make_tensor_value_info("input", DTYPE, GRID_SHAPE)
    y = onnx.helper.make_tensor_value_info("output", DTYPE, GRID_SHAPE)
    w = onnx.helper.make_tensor("W", DTYPE, list(weights.shape), weights.astype(np.float32).ravel().tolist())
    node = onnx.helper.make_node(
        "Conv",
        ["input", "W"],
        ["output"],
        kernel_shape=[kernel_size, kernel_size],
        pads=[kernel_size // 2] * 4,
    )
    graph = onnx.helper.make_graph([node], "graph", [x], [y], [w])
    model = onnx.helper.make_model(graph, ir_version=IR_VERSION, opset_imports=OPSET_IMPORTS)
    onnx.checker.check_model(model)
    return model


def make_identity_model() -> onnx.ModelProto:
    w = np.zeros((CHANNELS, CHANNELS, 1, 1), dtype=np.float32)
    for c in range(CHANNELS):
        w[c, c, 0, 0] = 1.0
    return make_conv_model(w, 1)


def make_zero_model() -> onnx.ModelProto:
    w = np.zeros((CHANNELS, CHANNELS, 1, 1), dtype=np.float32)
    return make_conv_model(w, 1)


def make_color_map_model(mapping: Dict[int, int]) -> onnx.ModelProto:
    w = np.zeros((CHANNELS, CHANNELS, 1, 1), dtype=np.float32)
    for in_color in range(CHANNELS):
        out_color = mapping.get(in_color, in_color)
        if out_color >= 0:
            w[out_color, in_color, 0, 0] = 1.0
    return make_conv_model(w, 1)


def make_gather_model(mode: str) -> onnx.ModelProto:
    x = onnx.helper.make_tensor_value_info("input", DTYPE, GRID_SHAPE)
    y = onnx.helper.make_tensor_value_info("output", DTYPE, GRID_SHAPE)
    ridx = np.arange(WIDTH - 1, -1, -1, dtype=np.int64)
    ridx_t = onnx.helper.make_tensor("RIDX", onnx.TensorProto.INT64, [WIDTH], ridx.tolist())

    if mode == "hflip":
        nodes = [onnx.helper.make_node("Gather", ["input", "RIDX"], ["output"], axis=3)]
        init = [ridx_t]
    elif mode == "vflip":
        nodes = [onnx.helper.make_node("Gather", ["input", "RIDX"], ["output"], axis=2)]
        init = [ridx_t]
    elif mode == "transpose":
        nodes = [onnx.helper.make_node("Transpose", ["input"], ["output"], perm=[0, 1, 3, 2])]
        init = []
    elif mode == "rot90":
        nodes = [
            onnx.helper.make_node("Transpose", ["input"], ["t1"], perm=[0, 1, 3, 2]),
            onnx.helper.make_node("Gather", ["t1", "RIDX"], ["output"], axis=2),
        ]
        init = [ridx_t]
    elif mode == "rot270":
        nodes = [
            onnx.helper.make_node("Transpose", ["input"], ["t1"], perm=[0, 1, 3, 2]),
            onnx.helper.make_node("Gather", ["t1", "RIDX"], ["output"], axis=3),
        ]
        init = [ridx_t]
    elif mode == "rot180":
        nodes = [
            onnx.helper.make_node("Gather", ["input", "RIDX"], ["t1"], axis=2),
            onnx.helper.make_node("Gather", ["t1", "RIDX"], ["output"], axis=3),
        ]
        init = [ridx_t]
    else:
        raise ValueError(mode)

    graph = onnx.helper.make_graph(nodes, f"graph_{mode}", [x], [y], init)
    model = onnx.helper.make_model(graph, ir_version=IR_VERSION, opset_imports=OPSET_IMPORTS)
    onnx.checker.check_model(model)
    return model


def run_session(session: ort.InferenceSession, x: np.ndarray) -> np.ndarray:
    y = session.run(["output"], {"input": x})[0]
    return (y > 0.0).astype(np.float32)


def is_correct_for_examples(session: ort.InferenceSession, task_payload: Dict) -> bool:
    for subset in ("train", "test", "arc-gen"):
        for example in task_payload.get(subset, []):
            x = one_hot_grid(example["input"])
            expected = one_hot_grid(example["output"])
            got = run_session(session, x)
            if not np.array_equal(got, expected):
                return False
    return True


def infer_color_mapping(task_payload: Dict) -> Tuple[bool, Dict[int, int]]:
    mapping: Dict[int, int] = {}
    for subset in ("train", "test", "arc-gen"):
        for example in task_payload.get(subset, []):
            x = one_hot_grid(example["input"])
            y = one_hot_grid(example["output"])
            for r in range(HEIGHT):
                for c in range(WIDTH):
                    xi = x[0, :, r, c]
                    yi = y[0, :, r, c]
                    if xi.sum() == 0.0:
                        if yi.sum() > 0.0:
                            return False, {}
                        continue
                    in_color = int(np.argmax(xi))
                    out_color = int(np.argmax(yi)) if yi.sum() > 0.0 else -1
                    if in_color in mapping and mapping[in_color] != out_color:
                        return False, {}
                    mapping[in_color] = out_color
    return True, mapping

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for old in OUTPUT_DIR.glob("task*.onnx"):
    old.unlink()

task_files = sorted(DATA_DIR.glob("task*.json"))
if LIMIT > 0:
    task_files = task_files[:LIMIT]

solved = []
written = []
selected_names = {}

def save_model(model: onnx.ModelProto, path: Path) -> None:
    onnx.save(model, path)

with tempfile.TemporaryDirectory() as td:
    tmp_dir = Path(td)

    static_builders = {
        "identity": make_identity_model,
        "zero": make_zero_model,
        "hflip": lambda: make_gather_model("hflip"),
        "vflip": lambda: make_gather_model("vflip"),
        "transpose": lambda: make_gather_model("transpose"),
        "rot90": lambda: make_gather_model("rot90"),
        "rot180": lambda: make_gather_model("rot180"),
        "rot270": lambda: make_gather_model("rot270"),
    }

    static_paths = {}
    static_sessions = {}
    for name, builder in static_builders.items():
        model = builder()
        path = tmp_dir / f"{name}.onnx"
        save_model(model, path)
        static_paths[name] = path
        static_sessions[name] = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])

    for task_file in task_files:
        task_name = task_file.stem
        with task_file.open("r", encoding="utf-8") as f:
            task_payload = json.load(f)

        chosen_name = None
        chosen_model = None

        static_order = ["identity", "hflip", "vflip", "transpose", "rot90", "rot180", "rot270", "zero"]
        for candidate_name in static_order:
            if is_correct_for_examples(static_sessions[candidate_name], task_payload):
                chosen_name = candidate_name
                chosen_model = onnx.load(static_paths[candidate_name])
                break

        ok_map, color_map = infer_color_mapping(task_payload)
        if chosen_name is None and ok_map:
            model = make_color_map_model(color_map)
            color_path = tmp_dir / f"{task_name}_color_map.onnx"
            save_model(model, color_path)
            color_session = ort.InferenceSession(str(color_path), providers=["CPUExecutionProvider"])
            if is_correct_for_examples(color_session, task_payload):
                chosen_name = "color_map"
                chosen_model = model

        if chosen_name is None and WRITE_ALL_TASKS:
            chosen_name = "identity"
            chosen_model = onnx.load(static_paths["identity"])

        if chosen_model is not None:
            out_path = OUTPUT_DIR / f"{task_name}.onnx"
            save_model(chosen_model, out_path)
            written.append(task_name)
            selected_names[task_name] = chosen_name

        if chosen_name is not None and chosen_name != "identity":
            solved.append(task_name)
        elif chosen_name == "identity" and is_correct_for_examples(static_sessions["identity"], task_payload):
            solved.append(task_name)

with zipfile.ZipFile(ZIP_NAME, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for onnx_file in sorted(OUTPUT_DIR.glob("task*.onnx")):
        zf.write(onnx_file, arcname=onnx_file.name)

from collections import Counter

counter = Counter(selected_names.values())
print(f"Processed tasks: {len(task_files)}")
print(f"Public solved tasks: {len(solved)}")
print(f"Networks written: {len(written)}")
print(f"Model usage: {dict(counter)}")
print(f"Wrote ONNX files to: {OUTPUT_DIR.resolve()}")
print(f"Wrote zip: {ZIP_NAME.resolve()}")

Processed tasks: 400
Public solved tasks: 6
Networks written: 400
Model usage: {'identity': 394, 'color_map': 4, 'transpose': 2}
Wrote ONNX files to: C:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\neurogolf-2026\baseline_submission
Wrote zip: C:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\neurogolf-2026\submission.zip
